# F260517 predicted-coordinate surfaces

This notebook loads the saved `phase_new_fXXXXXX.npy` files and renders the selected 10 surfaces as filled 3D grids. The `motion` mode keeps the frame-1 surface as a baseline and magnifies only the temporal z displacement for visual inspection.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm, colors
import matplotlib.colors as mcolors
from matplotlib.lines import Line2D

# Change these parameters to tune the visualization.
RESULT_DIR = Path('/home/cyf/wbi/Virginia/registrated_data/f260517/f260517_0820/diagnostics')
OUTPUT_DIR = RESULT_DIR / 'coords_visualization'
FRAME_LABELS = [1, 40, 80, 120, 160, 200]
FILE_FRAMES = [0, 39, 79, 119, 159, 199]  # saved files are zero-based
K_INDICES = list(range(0, 20, 2))  # 10 of the 20 available surfaces
FIXED_Z = np.arange(20.0, 210.1, 10.0, dtype=np.float32)  # exact pipeline fixed planes
DOWNSAMPLE_X = 25
DOWNSAMPLE_Y = 50
GRID_STRIDE = 2
ELEV = 25
AZIM = -58
Z_SCALE = 3.0  # visualization-only magnification around each fixed plane
MODE = 'motion'  # 'motion' or 'absolute'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_selected_coords():
    blocks = []
    for file_frame in FILE_FRAMES:
        path = RESULT_DIR / f'phase_new_f{file_frame:06d}.npy'
        phase = np.load(path, mmap_mode='r')
        # phase: (X, Y, K, 3) -> (K, sampled-Y, sampled-X, 3)
        sample = phase[::DOWNSAMPLE_X, ::DOWNSAMPLE_Y, K_INDICES, :]
        blocks.append(np.asarray(sample.transpose(2, 1, 0, 3), dtype=np.float32))
    return np.stack(blocks, axis=0)

coords = load_selected_coords()
print('loaded shape:', coords.shape)  # (6 frames, 10 surfaces, rows, cols, xyz)

In [ ]:
def _display_z(coords, mode='motion', z_scale=3.0):
    z = coords[..., 2]
    fixed = FIXED_Z[np.asarray(K_INDICES)][None, :, None, None]
    distance = np.abs(z - fixed)
    if mode == 'absolute':
        return z, distance, 'distance to fixed plane |Δz|'
    if mode == 'motion':
        return fixed + z_scale * (z - fixed), distance, 'distance to fixed plane |Δz|'
    raise ValueError("mode must be 'absolute' or 'motion'")

def render_frame(ax, coords_frame, z_frame, color_frame, norm, frame_label, mode):
    for surface_index, k in enumerate(K_INDICES):
        X = coords_frame[surface_index, ..., 0]
        Y = coords_frame[surface_index, ..., 1]
        Z = z_frame[surface_index]
        base_rgb = np.asarray(plt.get_cmap('tab10')(surface_index % 10)[:3])
        base_hsv = mcolors.rgb_to_hsv(base_rgb)
        intensity = norm(color_frame[surface_index])
        hsv = np.zeros((*intensity.shape, 3), dtype=np.float32)
        hsv[..., 0] = base_hsv[0]
        hsv[..., 1] = 0.30 + 0.70 * intensity
        hsv[..., 2] = 0.98 - 0.58 * intensity
        facecolors = mcolors.hsv_to_rgb(hsv)
        line_color = np.clip(base_rgb * 0.45, 0.0, 1.0)
        ax.plot_surface(X, Y, Z, facecolors=facecolors, linewidth=0.0,
                        antialiased=True, shade=False, alpha=0.55)
        ax.plot_wireframe(X, Y, Z, rstride=GRID_STRIDE, cstride=GRID_STRIDE,
                          color=line_color, linewidth=0.75, alpha=0.90)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.grid(False)
    ax.view_init(elev=ELEV, azim=AZIM)

def render_montage(coords, mode=MODE, z_scale=Z_SCALE, output_name=None):
    z_display, color_values, color_label = _display_z(coords, mode, z_scale)
    norm = colors.Normalize(vmin=0.0, vmax=float(np.nanmax(color_values)))
    fig = plt.figure(figsize=(18, 10), constrained_layout=True)
    axes = [fig.add_subplot(2, 3, i + 1, projection='3d') for i in range(len(FRAME_LABELS))]
    x_all = coords[..., 0]
    y_all = coords[..., 1]
    for ax, i, label in zip(axes, range(len(FRAME_LABELS)), FRAME_LABELS):
        render_frame(ax, coords[i], z_display[i], color_values[i], norm, label, mode)
        ax.set_xlim(float(np.nanmin(x_all)), float(np.nanmax(x_all)))
        ax.set_ylim(float(np.nanmin(y_all)), float(np.nanmax(y_all)))
        ax.set_zlim(float(np.nanmin(z_display)), float(np.nanmax(z_display)))
        ax.set_box_aspect((1.0, 1.7, 0.9))
    fig.suptitle(f'F260517 predicted coordinate surfaces ({mode}, z display scale={z_scale:g}x)')
    if output_name is None:
        output_name = f'coords_{mode}_montage_z{z_scale:g}x.png'
    output_path = OUTPUT_DIR / output_name
    fig.savefig(output_path, dpi=180, bbox_inches='tight')
    plt.show()
    return output_path

montage_path = render_montage(coords)
print(montage_path)

In [ ]:
def render_individual_frames(coords, mode=MODE, z_scale=Z_SCALE):
    z_display, color_values, _ = _display_z(coords, mode, z_scale)
    norm = colors.Normalize(vmin=0.0, vmax=float(np.nanmax(color_values)))
    paths = []
    for i, label in enumerate(FRAME_LABELS):
        fig = plt.figure(figsize=(10, 8), constrained_layout=True)
        ax = fig.add_subplot(111, projection='3d')
        render_frame(ax, coords[i], z_display[i], color_values[i], norm, label, mode)
        ax.set_xlim(float(np.nanmin(coords[..., 0])), float(np.nanmax(coords[..., 0])))
        ax.set_ylim(float(np.nanmin(coords[..., 1])), float(np.nanmax(coords[..., 1])))
        ax.set_zlim(float(np.nanmin(z_display)), float(np.nanmax(z_display)))
        ax.set_box_aspect((1.0, 1.7, 0.9))
        path = OUTPUT_DIR / f'coords_{mode}_frame{label:03d}_z{z_scale:g}x.png'
        fig.savefig(path, dpi=220, bbox_inches='tight')
        plt.close(fig)
        paths.append(path)
    return paths

def render_slice_legend():
    handles = [Line2D([0], [0], color=plt.get_cmap('tab10')(i % 10), lw=4,
                      label=f'k={k}, fixed z={FIXED_Z[k]:g}')
               for i, k in enumerate(K_INDICES)]
    fig, ax = plt.subplots(figsize=(10, 2.0))
    ax.axis('off')
    ax.legend(handles=handles, loc='center', ncol=5, frameon=False)
    path = OUTPUT_DIR / 'coords_slice_legend.png'
    fig.savefig(path, dpi=220, bbox_inches='tight', transparent=True)
    plt.close(fig)
    return path

individual_paths = render_individual_frames(coords)
legend_path = render_slice_legend()
individual_paths, legend_path